In [2]:
print("Hello, Jupyter!")

Hello, Jupyter!


In [2]:
import sys
import os
import platform

# Python 버전 확인
print("Python Version:", sys.version)

# 현재 작업 디렉토리 확인
print("Current Working Directory:", os.getcwd())

# 운영 체제 및 아키텍처 확인
print("OS Platform:", platform.system())
print("OS Architecture:", platform.machine())

# 설치된 패키지 목록 확인
!pip list

Python Version: 3.6.9 (default, Mar 15 2022, 13:55:28) 
[GCC 8.4.0]
Current Working Directory: /home/soda/Project/python/notebook
OS Platform: Linux
OS Architecture: aarch64
Package                       Version
----------------------------- -------------------
absl-py                       0.12.0
aenum                         3.1.11
aiohttp                       3.8.1
aiosignal                     1.2.0
alabaster                     0.7.12
anyio                         3.5.0
appdirs                       1.4.4
apturl                        0.5.2
argon2-cffi                   21.3.0
argon2-cffi-bindings          21.2.0
asn1crypto                    0.24.0
astroid                       2.11.2
astunparse                    1.6.3
async-generator               1.10
async-timeout                 4.0.2
asynctest                     0.13.0
attrs                         21.4.0
audioread                     2.1.9
autobahn                      21.2.1
Babel                         2.9.1
backcall 

In [3]:
import cv2
from pop import Camera
from pop import Util
cam = Camera(300,300)
Util.enable_imshow()

while True:
    img = cam.value
    Util.imshow("frame",img)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

KeyboardInterrupt: 

In [1]:
#카메라 촬영
import cv2
from pop import Camera
from pop import Util

# 카메라 초기화 (300x300 해상도)
cam = Camera(300, 300)

# imshow 활성화
Util.enable_imshow()

# 전역 변수로 프레임 저장
global_frame = None

while True:
    # 카메라에서 프레임 읽기
    img = cam.value

    # 전역 변수에 프레임 저장
    global_frame = img

    # 현재 프레임을 "Camera Feed" 창에 표시
    Util.imshow("Camera Feed", img)

    # 키 입력 대기 (1ms)
    key = cv2.waitKey(1) & 0xFF

    # 'q' 키를 누르면 종료
    if key == ord('q'):
        break

# 자원 해제 및 창 닫기
cv2.destroyAllWindows()

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

KeyboardInterrupt: 

In [6]:
#일단 스샷 한장 되는걸루
import cv2
from pop import Camera
from datetime import datetime
import os

# 카메라 초기화
cam = Camera(300, 300)

# 저장할 디렉토리 설정
save_directory = "dataset2/"

# 디렉토리가 없다면 생성
os.makedirs(save_directory, exist_ok=True)

# 현재 프레임 가져오기
img = cam.value

# 현재 시간을 타임스탬프 형식으로 변환
timestamp = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
filename = f"{save_directory}capture_{timestamp}.jpg"

# 이미지 저장
cv2.imwrite(filename, img)
print(f"Image saved: {filename}")

# 저장된 이미지를 주피터 노트북에 표시
from IPython.display import display
import ipywidgets as widgets
display(widgets.Image(value=cv2.imencode('.jpg', img)[1].tobytes(), format='jpg'))

Image saved: dataset2/capture_2025-03-31_17:45:13.jpg


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [2]:
#완벽 실시간 동영상
import cv2
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
from pop import Camera
from IPython.display import display
import ipywidgets as widgets

# 디바이스 설정 (GPU 사용 가능 여부 확인)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 전역 변수: 상태 값 (0: free, 1: blocked)
status_value = 0  # 초기값은 free 상태

# 모델 정의
class MyTransferLearningModel(torch.nn.Module):
    def __init__(self, pretrained_model, feature_extractor):
        super().__init__()
        if feature_extractor:
            for param in pretrained_model.parameters():
                param.requires_grad = False
        pretrained_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(pretrained_model.classifier[1].in_features, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(64, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 2)  # 2가지 분류 (blocked, free)
        )
        self.model = pretrained_model
        
    def forward(self, data):
        logits = self.model(data)
        return logits

# 사전 학습된 MobileNetV2 모델 로드
pretrained_model = models.mobilenet_v2(pretrained=True)

# Feature Extractor 여부 설정
feature_extractor = True

# 모델 인스턴스 생성
model = MyTransferLearningModel(pretrained_model, feature_extractor).to(DEVICE)

# 저장된 가중치 파일 경로 (Jetson 환경에 맞게 수정)
MODEL_PATH = "/home/soda/Project/python/notebook/dataset2/MobileNetV2_best_model3.pth"

# 모델 가중치 로드
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()  # 모델을 평가 모드로 설정

# 입력 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNetV2는 224x224 입력을 기대합니다
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
])

# 클래스 레이블 정의
class_labels = ["blocked", "free"]

# 상태 값을 숫자로 변환하는 함수
def get_status_value(state):
    return 1 if state == "blocked" else 0

# 상태 업데이트 함수
def update_status(frame):
    global last_state, current_state, state_start_time, status_value

    # 모델 입력을 위한 전처리
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # OpenCV 이미지를 PIL 이미지로 변환
    input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)  # 배치 차원 추가 및 GPU로 이동

    # 모델 예측
    with torch.no_grad():  # 그래디언트 계산 비활성화
        outputs = model(input_tensor)
        _, predicted = torch.max(outputs, 1)  # 가장 높은 확률의 클래스 선택

    # 예측 결과 가져오기
    result = class_labels[predicted.item()]

    # 상태 변화 체크
    if result != current_state:
        current_state = result
        state_start_time = time.time()  # 상태 시작 시간 갱신

    # 상태가 유지된 시간 계산
    elapsed_time = time.time() - state_start_time

    # 상태가 0.5초 이상 유지되었을 때만 최종 결과로 간주
    if elapsed_time >= STATE_DURATION_THRESHOLD:
        last_state = current_state

    # 상태 값을 숫자로 변환 (0: free, 1: blocked)
    status_value = get_status_value(last_state)

    return status_value, last_state

# 카메라 초기화 (224x224 해상도)
cam = Camera(224, 224)

# 상태와 시간 초기화
last_state = None  # 이전 상태
current_state = None  # 현재 상태
state_start_time = time.time()  # 상태 시작 시간
STATE_DURATION_THRESHOLD = 0.5  # 상태가 유지되어야 하는 최소 시간 (초)

# Jupyter Notebook에서 이미지를 표시하기 위한 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

print("실시간 처리 시작... ('q' 키를 누르면 종료)")

while True:
    # 현재 프레임 읽기
    frame = cam.value
    if frame is None:
        print("프레임을 읽을 수 없습니다.")
        break

    # 상태 업데이트 및 값 반환
    status_value, last_state = update_status(frame)

    # 상태를 화면 상단에 표시
    cv2.putText(frame, f"Status: {last_state} ({status_value})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Jupyter Notebook에서 프레임 표시
    image_widget.value = cv2.imencode('.jpg', frame)[1].tobytes()

    # 'q' 키를 누르면 종료
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

# 자원 해제
cv2.destroyAllWindows()

Image(value=b'', format='jpeg', height='224', width='224')

실시간 처리 시작... ('q' 키를 누르면 종료)


KeyboardInterrupt: 

In [1]:
#카메라 실시간 구동, 시험 준비 완료
import cv2
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
from pop import Camera
from IPython.display import display
import ipywidgets as widgets
from threading import Thread
from multiprocessing import Value
import ctypes

# 디바이스 설정 (GPU 사용 가능 여부 확인)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 전역 변수: 상태 값 (0: free, 1: blocked)
# 공유 메모리 객체를 사용하여 상태 저장
status_shared = Value(ctypes.c_int, 0)  # 초기값은 free 상태

# 모델 정의
class MyTransferLearningModel(torch.nn.Module):
    def __init__(self, pretrained_model, feature_extractor):
        super().__init__()
        if feature_extractor:
            for param in pretrained_model.parameters():
                param.requires_grad = False
        pretrained_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(pretrained_model.classifier[1].in_features, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(64, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 2)  # 2가지 분류 (blocked, free)
        )
        self.model = pretrained_model
        
    def forward(self, data):
        logits = self.model(data)
        return logits

# 사전 학습된 MobileNetV2 모델 로드
pretrained_model = models.mobilenet_v2(pretrained=True)

# Feature Extractor 여부 설정
feature_extractor = True

# 모델 인스턴스 생성
model = MyTransferLearningModel(pretrained_model, feature_extractor).to(DEVICE)

# 저장된 가중치 파일 경로 (Jetson 환경에 맞게 수정)
MODEL_PATH = "/home/soda/Project/python/notebook/dataset2/MobileNetV2_best_model3.pth"

# 모델 가중치 로드
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()  # 모델을 평가 모드로 설정

# 입력 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNetV2는 224x224 입력을 기대합니다
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
])

# 클래스 레이블 정의
class_labels = ["blocked", "free"]

# 상태 값을 숫자로 변환하는 함수
def get_status_value(state):
    return 1 if state == "blocked" else 0

# 상태 업데이트 함수
def update_status(frame):
    global last_state, current_state, state_start_time, status_shared

    # 모델 입력을 위한 전처리
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # OpenCV 이미지를 PIL 이미지로 변환
    input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)  # 배치 차원 추가 및 GPU로 이동

    # 모델 예측
    with torch.no_grad():  # 그래디언트 계산 비활성화
        outputs = model(input_tensor)
        _, predicted = torch.max(outputs, 1)  # 가장 높은 확률의 클래스 선택

    # 예측 결과 가져오기
    result = class_labels[predicted.item()]

    # 상태 변화 체크
    if result != current_state:
        current_state = result
        state_start_time = time.time()  # 상태 시작 시간 갱신

    # 상태가 유지된 시간 계산
    elapsed_time = time.time() - state_start_time

    # 상태가 0.5초 이상 유지되었을 때만 최종 결과로 간주
    if elapsed_time >= STATE_DURATION_THRESHOLD:
        last_state = current_state

    # 상태 값을 숫자로 변환하고 공유 메모리에 업데이트 (0: free, 1: blocked)
    status_value = get_status_value(last_state)
    with status_shared.get_lock():
        status_shared.value = status_value

    return status_value, last_state

# 카메라 초기화 (224x224 해상도)
cam = Camera(224, 224)

# 상태와 시간 초기화
last_state = "free"  # 이전 상태 초기값 설정
current_state = "free"  # 현재 상태 초기값 설정
state_start_time = time.time()  # 상태 시작 시간
STATE_DURATION_THRESHOLD = 0.5  # 상태가 유지되어야 하는 최소 시간 (초)

# Jupyter Notebook에서 이미지를 표시하기 위한 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

# 트리거 상태 초기화
trigger_active = False  # 트리거 활성화 여부
last_trigger_time = time.time()  # 마지막 트리거 출력 시간

# 버튼 위젯 생성
trigger_button = widgets.ToggleButton(
    value=False,
    description="Trigger",
    button_style="success",  # 'success', 'info', 'warning', 'danger' 중 선택
    tooltip="Toggle trigger on/off"
)

# 상태 출력 텍스트 위젯 생성
status_output = widgets.Text(
    value='Status: 0 (free)',
    description='현재 상태:',
    disabled=True
)
display(status_output)

# 버튼 클릭 시 트리거 상태 변경
def on_button_click(change):
    global trigger_active
    trigger_active = change.new
    if trigger_active:
        print("트리거 활성화됨 (1초마다 상태 출력)")
    else:
        print("트리거 비활성화됨")

trigger_button.observe(on_button_click, names="value")
display(trigger_button)

print("실시간 처리 시작...")

# 현재 상태값을 가져오는 함수 정의 (다른 셀에서 사용 가능)
def get_current_status():
    with status_shared.get_lock():
        return status_shared.value

def camera_loop():
    global trigger_active, last_trigger_time
    while True:
        # 현재 프레임 읽기
        frame = cam.value
        if frame is None:
            print("프레임을 읽을 수 없습니다.")
            break

        # 상태 업데이트 및 값 반환
        status_value, last_state = update_status(frame)

        # 상태를 화면 상단에 표시
        cv2.putText(frame, f"Status: {last_state} ({status_value})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # 상태 텍스트 위젯 업데이트
        status_output.value = f'Status: {status_value} ({last_state})'

        # Jupyter Notebook에서 프레임 표시
        image_widget.value = cv2.imencode('.jpg', frame)[1].tobytes()

        # 트리거 상태에 따른 자동 출력
        if trigger_active and time.time() - last_trigger_time >= 1.0:  # 1초마다 상태 출력
            current_status = get_current_status()
            print(f"현재 상태: {current_status} ({last_state})")
            last_trigger_time = time.time()

# 별도 스레드에서 카메라 루프 실행
camera_thread = Thread(target=camera_loop)
camera_thread.daemon = True
camera_thread.start()

Image(value=b'', format='jpeg', height='224', width='224')

Text(value='Status: 0 (free)', description='현재 상태:', disabled=True)

ToggleButton(value=False, button_style='success', description='Trigger', tooltip='Toggle trigger on/off')

실시간 처리 시작...


In [2]:
# 현재 상태값 확인
current_status = get_current_status()
print(f"현재 상태값: {current_status}")

현재 상태값: 0
현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 1 (blocked)
현재 상태: 1 (blocked)
현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 0 (free)


In [5]:
from pop import Pilot
autocar = Pilot.AutoCar()
import time

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    for _ in range(1):  # 1번 반복 (8자를 그리기 위해)
        # 왼쪽으로 회전 (왼쪽 조향 + 전진)
        autocar.steering = -1  # 왼쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
        time.sleep(6)          # 6초 동안 유지
        
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진
        time.sleep(2)          # 2초 동안 유지
        
        # 오른쪽으로 회전 (오른쪽 조향 + 전진)
        autocar.steering = 1   # 오른쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
        time.sleep(6)          # 6초 동안 유지
        
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진
        time.sleep(1)          # 1초 동안 유지
    
    # 정지
    autocar.steering = 0      # 중앙으로 조향
    autocar.stop()            # 정지

# 실행
dance_in_8()

KeyboardInterrupt: 

In [2]:
#자동차 주행 버전1
from pop import Pilot
import time
from threading import Thread
import ipywidgets as widgets
from IPython.display import display

# 자동차 초기화
autocar = Pilot.AutoCar()

# 시작 버튼 위젯 생성
start_button = widgets.Button(
    description="8자 주행 시작",
    button_style="success",
    tooltip="8자 주행을 시작합니다"
)
display(start_button)

# 주행 상태 표시 위젯
driving_status = widgets.Text(
    value='상태: 대기 중',
    description='주행 상태:',
    disabled=True
)
display(driving_status)

# 주행 중인지 확인하는 플래그
is_driving = False

# 자동차 상태 제어 함수 - 0.5초마다 실행
def control_car_by_status():
    global is_driving
    
    while True:
        # 0.5초마다 상태 확인
        time.sleep(0.5)
        
        # 현재 카메라 상태값 확인 (첫 번째 셀에서 정의된 함수 사용)
        current_status = get_current_status()
        
        # 주행 중일 때만 상태에 따라 제어
        if is_driving:
            if current_status == 1:  # blocked
                print("장애물 감지! 차량 정지")
                autocar.steering = 0
                autocar.stop()
                is_driving = False
                driving_status.value = '상태: 장애물 감지로 정지'

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    global is_driving
    
    # 주행 상태 설정
    is_driving = True
    driving_status.value = '상태: 8자 주행 중'
    
    try:
        for _ in range(1):  # 1번 반복 (8자를 그리기 위해)
            # 차량 주행 중 blocked 상태를 감지하면 스레드에서 차량이 정지됨
            
            # 왼쪽으로 회전 (왼쪽 조향 + 전진)
            autocar.steering = -1  # 왼쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(6)          # 6초 동안 유지
            
            # 현재 주행 중인지 확인
            if not is_driving:
                return
            
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(2)          # 2초 동안 유지
            
            # 현재 주행 중인지 확인
            if not is_driving:
                return
            
            # 오른쪽으로 회전 (오른쪽 조향 + 전진)
            autocar.steering = 1   # 오른쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(6)          # 6초 동안 유지
            
            # 현재 주행 중인지 확인
            if not is_driving:
                return
            
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(1)          # 1초 동안 유지
            
            # 현재 주행 중인지 확인
            if not is_driving:
                return
        
        # 8자 주행 완료 후 정지
        autocar.steering = 0      # 중앙으로 조향
        autocar.stop()            # 정지
        is_driving = False
        driving_status.value = '상태: 8자 주행 완료'
        
    except Exception as e:
        print(f"오류 발생: {e}")
        autocar.stop()  # 오류 발생 시 안전하게 정지
        is_driving = False
        driving_status.value = '상태: 오류 발생'

def on_start_click(b):
    # 이미 주행 중이면 무시
    global is_driving
    if is_driving:
        print("이미 주행 중입니다.")
        return
        
    # 먼저 현재 상태 확인
    current_status = get_current_status()
    if current_status == 1:  # blocked
        print("장애물이 감지되었습니다. 주행을 시작할 수 없습니다.")
        driving_status.value = '상태: 장애물 감지됨'
        return
    
    # 별도의 스레드에서 주행 함수 실행
    drive_thread = Thread(target=dance_in_8)
    drive_thread.daemon = True
    drive_thread.start()
    
start_button.on_click(on_start_click)

# 자동차 제어 스레드 시작
car_control_thread = Thread(target=control_car_by_status)
car_control_thread.daemon = True
car_control_thread.start()

print("자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.")

현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 0 (free)


Button(button_style='success', description='8자 주행 시작', style=ButtonStyle(), tooltip='8자 주행을 시작합니다')

Text(value='상태: 대기 중', description='주행 상태:', disabled=True)

자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.
현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 0 (free)
현재 상태: 0 (free)


In [3]:
#자동차 버전2
from pop import Pilot
import time
from threading import Thread, Event
import ipywidgets as widgets
from IPython.display import display

# 자동차 초기화
autocar = Pilot.AutoCar()

# 시작 버튼 위젯 생성
start_button = widgets.Button(
    description="8자 주행 시작",
    button_style="success",
    tooltip="8자 주행을 시작합니다"
)
display(start_button)

# 주행 상태 표시 위젯
driving_status = widgets.Text(
    value='상태: 대기 중',
    description='주행 상태:',
    disabled=True
)
display(driving_status)

# 주행 중인지 확인하는 플래그
is_driving = False

# 일시정지를 위한 이벤트
pause_event = Event()
pause_event.set()  # 초기값은 정지 안함 (실행 가능)

# 자동차 상태 제어 함수 - 0.5초마다 실행
def control_car_by_status():
    global is_driving
    
    while True:
        # 0.5초마다 상태 확인
        time.sleep(0.5)
        
        # 현재 카메라 상태값 확인 (첫 번째 셀에서 정의된 함수 사용)
        current_status = get_current_status()
        
        # 주행 중일 때만 상태에 따라 제어
        if is_driving:
            if current_status == 1:  # blocked
                # 일시정지
                if pause_event.is_set():
                    print("장애물 감지! 주행 일시정지")
                    autocar.stop()  # 자동차 정지
                    pause_event.clear()  # 이벤트 해제로 주행 함수 일시정지
                    driving_status.value = '상태: 장애물 감지로 일시정지'
            else:  # free
                # 재개
                if not pause_event.is_set():
                    print("장애물 제거! 주행 재개")
                    pause_event.set()  # 이벤트 설정으로 주행 함수 재개
                    driving_status.value = '상태: 주행 재개'

# 안전한 sleep 함수 (일시정지 가능)
def safe_sleep(seconds):
    """일시정지 가능한 sleep 함수"""
    end_time = time.time() + seconds
    
    while time.time() < end_time:
        # 일시정지 여부 확인
        if not pause_event.is_set():
            # 일시정지 상태이면 wait
            print("일시정지 상태입니다. 장애물이 제거될 때까지 대기...")
            pause_event.wait()  # 이벤트가 설정될 때까지 대기
            # 재개된 시점에서 남은 시간 재계산
            end_time = time.time() + (end_time - time.time())
        
        # 0.1초씩 대기하면서 중간중간 확인
        time.sleep(0.1)

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    global is_driving
    
    # 주행 상태 설정
    is_driving = True
    driving_status.value = '상태: 8자 주행 중'
    
    try:
        for _ in range(1):  # 1번 반복 (8자를 그리기 위해)
            # 왼쪽으로 회전 (왼쪽 조향 + 전진)
            print("왼쪽으로 회전 시작")
            autocar.steering = -1  # 왼쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            
            # 안전한 sleep 사용 (일시정지 가능)
            safe_sleep(6)
            
            # 직진
            print("직진 시작")
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            
            # 안전한 sleep 사용
            safe_sleep(2)
            
            # 오른쪽으로 회전 (오른쪽 조향 + 전진)
            print("오른쪽으로 회전 시작")
            autocar.steering = 1   # 오른쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            
            # 안전한 sleep 사용
            safe_sleep(6)
            
            # 직진
            print("마지막 직진 시작")
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            
            # 안전한 sleep 사용
            safe_sleep(1)
        
        # 8자 주행 완료 후 정지
        print("8자 주행 완료")
        autocar.steering = 0      # 중앙으로 조향
        autocar.stop()            # 정지
        is_driving = False
        driving_status.value = '상태: 8자 주행 완료'
        
    except Exception as e:
        print(f"오류 발생: {e}")
        autocar.stop()  # 오류 발생 시 안전하게 정지
        is_driving = False
        driving_status.value = '상태: 오류 발생'

def on_start_click(b):
    # 이미 주행 중이면 무시
    global is_driving
    if is_driving:
        print("이미 주행 중입니다.")
        return
        
    # 먼저 현재 상태 확인
    current_status = get_current_status()
    if current_status == 1:  # blocked
        print("장애물이 감지되었습니다. 주행을 시작할 수 없습니다.")
        driving_status.value = '상태: 장애물 감지됨'
        return
    
    # pause_event 초기화
    pause_event.set()
    
    # 별도의 스레드에서 주행 함수 실행
    drive_thread = Thread(target=dance_in_8)
    drive_thread.daemon = True
    drive_thread.start()
    
start_button.on_click(on_start_click)

# 자동차 제어 스레드 시작
car_control_thread = Thread(target=control_car_by_status)
car_control_thread.daemon = True
car_control_thread.start()

print("자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.")

현재 상태: 0 (free)


Button(button_style='success', description='8자 주행 시작', style=ButtonStyle(), tooltip='8자 주행을 시작합니다')

Text(value='상태: 대기 중', description='주행 상태:', disabled=True)

자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.
현재 상태: 0 (free)


In [3]:
#긴급 제동 일시정지 개선

from pop import Pilot
import time
from threading import Thread, Event
import ipywidgets as widgets
from IPython.display import display

# 자동차 초기화
autocar = Pilot.AutoCar()

# 시작 버튼 위젯 생성
start_button = widgets.Button(
    description="8자 주행 시작",
    button_style="success",
    tooltip="8자 주행을 시작합니다"
)
display(start_button)

# 주행 상태 표시 위젯
driving_status = widgets.Text(
    value='상태: 대기 중',
    description='주행 상태:',
    disabled=True
)
display(driving_status)

# 주행 중인지 확인하는 플래그
is_driving = False

# 일시정지를 위한 이벤트
pause_event = Event()
pause_event.set()  # 초기값은 정지 안함 (실행 가능)

# 주행 단계를 관리하는 변수
driving_stage = 0
stage_start_time = 0

# 각 단계별 지속 시간 (초)
stage_durations = [
    6,   # 0: 왼쪽 회전
    2,   # 1: 직진
    6,   # 2: 오른쪽 회전
    1    # 3: 마지막 직진
]

# 각 단계별 동작 함수
def stage_actions(stage):
    if stage == 0:  # 왼쪽 회전
        print("왼쪽으로 회전 시작")
        autocar.steering = -1  # 왼쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 1:  # 직진
        print("직진 시작")
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 2:  # 오른쪽 회전
        print("오른쪽으로 회전 시작")
        autocar.steering = 1   # 오른쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 3:  # 마지막 직진
        print("마지막 직진 시작")
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진

# 자동차 상태 제어 함수 - 0.1초마다 실행
def control_car_by_status():
    global is_driving, driving_stage, stage_start_time
    
    while True:
        time.sleep(0.1)  # 더 자주 체크하도록 변경
        
        # 현재 카메라 상태값 확인
        current_status = get_current_status()
        
        # 주행 중일 때만 상태에 따라 제어
        if is_driving:
            if current_status == 1:  # blocked
                # 일시정지
                if pause_event.is_set():
                    print("장애물 감지! 주행 일시정지")
                    autocar.stop()  # 자동차 정지
                    pause_event.clear()  # 이벤트 해제로 주행 함수 일시정지
                    driving_status.value = '상태: 장애물 감지로 일시정지'
            else:  # free
                # 재개
                if not pause_event.is_set():
                    print("장애물 제거! 주행 재개")
                    pause_event.set()  # 이벤트 설정으로 주행 함수 재개
                    # 현재 단계 동작 재시작
                    if driving_stage < len(stage_durations):
                        stage_actions(driving_stage)
                    driving_status.value = '상태: 주행 재개'
            
            # 주행 단계 관리 (일시정지 상태가 아닌 경우만)
            if pause_event.is_set() and driving_stage < len(stage_durations):
                elapsed_time = time.time() - stage_start_time
                
                # 현재 단계의 시간이 완료되면 다음 단계로
                if elapsed_time >= stage_durations[driving_stage]:
                    driving_stage += 1
                    
                    # 다음 단계 시작
                    if driving_stage < len(stage_durations):
                        stage_start_time = time.time()
                        stage_actions(driving_stage)
                    else:
                        # 모든 단계 완료
                        print("8자 주행 완료")
                        autocar.steering = 0      # 중앙으로 조향
                        autocar.stop()            # 정지
                        is_driving = False
                        driving_status.value = '상태: 8자 주행 완료'

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    global is_driving, driving_stage, stage_start_time
    
    # 주행 상태 설정
    is_driving = True
    driving_status.value = '상태: 8자 주행 중'
    
    try:
        # 첫 번째 단계 시작
        driving_stage = 0
        stage_start_time = time.time()
        stage_actions(driving_stage)
        
    except Exception as e:
        print(f"오류 발생: {e}")
        autocar.stop()  # 오류 발생 시 안전하게 정지
        is_driving = False
        driving_status.value = '상태: 오류 발생'

def on_start_click(b):
    # 이미 주행 중이면 무시
    global is_driving
    if is_driving:
        print("이미 주행 중입니다.")
        return
        
    # 먼저 현재 상태 확인
    current_status = get_current_status()
    if current_status == 1:  # blocked
        print("장애물이 감지되었습니다. 주행을 시작할 수 없습니다.")
        driving_status.value = '상태: 장애물 감지됨'
        return
    
    # pause_event 초기화
    pause_event.set()
    
    # 별도의 스레드에서 주행 함수 실행
    drive_thread = Thread(target=dance_in_8)
    drive_thread.daemon = True
    drive_thread.start()
    
start_button.on_click(on_start_click)

# 자동차 제어 스레드 시작
car_control_thread = Thread(target=control_car_by_status)
car_control_thread.daemon = True
car_control_thread.start()

print("자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.")




Button(button_style='success', description='8자 주행 시작', style=ButtonStyle(), tooltip='8자 주행을 시작합니다')

Text(value='상태: 대기 중', description='주행 상태:', disabled=True)

자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.


In [4]:
#차량 주행 시험 완료
from pop import Pilot
import time
from threading import Thread, Event
import ipywidgets as widgets
from IPython.display import display

# 자동차 초기화
autocar = Pilot.AutoCar()

# 버튼 생성 (가로로 나란히 표시하기 위한 레이아웃)
button_layout = widgets.Layout(width='150px', margin='0 10px 0 0')

# 시작 버튼 위젯 생성
start_button = widgets.Button(
    description="8자 주행 시작",
    button_style="success",
    tooltip="8자 주행을 시작합니다",
    layout=button_layout
)

# 정지 버튼 위젯 생성
stop_button = widgets.Button(
    description="주행 정지",
    button_style="danger",
    tooltip="주행을 정지하고 초기화합니다",
    layout=button_layout
)

# 버튼들을 가로로 배치
button_box = widgets.HBox([start_button, stop_button])
display(button_box)

# 주행 상태 표시 위젯
driving_status = widgets.Text(
    value='상태: 대기 중',
    description='주행 상태:',
    disabled=True
)
display(driving_status)

# 주행 중인지 확인하는 플래그
is_driving = False

# 일시정지를 위한 이벤트
pause_event = Event()
pause_event.set()  # 초기값은 정지 안함 (실행 가능)

# 주행 단계를 관리하는 변수
driving_stage = 0
stage_start_time = 0

# 각 단계별 지속 시간 (초)
stage_durations = [
    6,   # 0: 왼쪽 회전
    2,   # 1: 직진
    6,   # 2: 오른쪽 회전
    1    # 3: 마지막 직진
]

# 각 단계별 동작 함수
def stage_actions(stage):
    if stage == 0:  # 왼쪽 회전
        print("왼쪽으로 회전 시작")
        autocar.steering = -1  # 왼쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 1:  # 직진
        print("직진 시작")
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 2:  # 오른쪽 회전
        print("오른쪽으로 회전 시작")
        autocar.steering = 1   # 오른쪽으로 최대 조향
        autocar.forward(50)    # 50% 속도로 전진
    elif stage == 3:  # 마지막 직진
        print("마지막 직진 시작")
        autocar.steering = 0  # 직진
        autocar.forward(50)    # 50% 속도로 전진

# 시스템 초기화 함수
def reset_system():
    global is_driving, driving_stage, stage_start_time
    
    # 자동차 정지
    autocar.steering = 0
    autocar.stop()
    
    # 변수 초기화
    is_driving = False
    driving_stage = 0
    stage_start_time = 0
    
    # 일시정지 이벤트 초기화
    pause_event.set()
    
    # 상태 표시 업데이트
    driving_status.value = '상태: 대기 중'
    
    print("시스템이 초기화되었습니다. 다시 주행을 시작할 수 있습니다.")

# 자동차 상태 제어 함수 - 0.1초마다 실행
def control_car_by_status():
    global is_driving, driving_stage, stage_start_time
    
    while True:
        time.sleep(0.1)  # 더 자주 체크하도록 변경
        
        # 현재 카메라 상태값 확인
        current_status = get_current_status()
        
        # 주행 중일 때만 상태에 따라 제어
        if is_driving:
            if current_status == 1:  # blocked
                # 일시정지
                if pause_event.is_set():
                    print("장애물 감지! 주행 일시정지")
                    autocar.stop()  # 자동차 정지
                    pause_event.clear()  # 이벤트 해제로 주행 함수 일시정지
                    driving_status.value = '상태: 장애물 감지로 일시정지'
            else:  # free
                # 재개
                if not pause_event.is_set():
                    print("장애물 제거! 주행 재개")
                    pause_event.set()  # 이벤트 설정으로 주행 함수 재개
                    # 현재 단계 동작 재시작
                    if driving_stage < len(stage_durations):
                        stage_actions(driving_stage)
                    driving_status.value = '상태: 주행 재개'
            
            # 주행 단계 관리 (일시정지 상태가 아닌 경우만)
            if pause_event.is_set() and driving_stage < len(stage_durations):
                elapsed_time = time.time() - stage_start_time
                
                # 현재 단계의 시간이 완료되면 다음 단계로
                if elapsed_time >= stage_durations[driving_stage]:
                    driving_stage += 1
                    
                    # 다음 단계 시작
                    if driving_stage < len(stage_durations):
                        stage_start_time = time.time()
                        stage_actions(driving_stage)
                    else:
                        # 모든 단계 완료
                        print("8자 주행 완료")
                        reset_system()

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    global is_driving, driving_stage, stage_start_time
    
    # 주행 상태 설정
    is_driving = True
    driving_status.value = '상태: 8자 주행 중'
    
    try:
        # 첫 번째 단계 시작
        driving_stage = 0
        stage_start_time = time.time()
        stage_actions(driving_stage)
        
    except Exception as e:
        print(f"오류 발생: {e}")
        reset_system()  # 오류 발생 시 안전하게 초기화

def on_start_click(b):
    # 이미 주행 중이면 무시
    global is_driving
    if is_driving:
        print("이미 주행 중입니다.")
        return
        
    # 먼저 현재 상태 확인
    current_status = get_current_status()
    if current_status == 1:  # blocked
        print("장애물이 감지되었습니다. 주행을 시작할 수 없습니다.")
        driving_status.value = '상태: 장애물 감지됨'
        return
    
    # pause_event 초기화
    pause_event.set()
    
    # 별도의 스레드에서 주행 함수 실행
    drive_thread = Thread(target=dance_in_8)
    drive_thread.daemon = True
    drive_thread.start()

def on_stop_click(b):
    print("주행을 정지하고 시스템을 초기화합니다.")
    reset_system()
    
# 버튼 클릭 이벤트 연결
start_button.on_click(on_start_click)
stop_button.on_click(on_stop_click)

# 자동차 제어 스레드 시작
car_control_thread = Thread(target=control_car_by_status)
car_control_thread.daemon = True
car_control_thread.start()

print("자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.")
print("주행 중 언제든지 '주행 정지' 버튼을 눌러 주행을 중단할 수 있습니다.")

Text(value='상태: 대기 중', description='주행 상태:', disabled=True)

자동차 제어 시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.
주행 중 언제든지 '주행 정지' 버튼을 눌러 주행을 중단할 수 있습니다.


In [6]:
# 하나의 셀로. 필요 xx
import cv2
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
from pop import Camera
from pop import Pilot
from IPython.display import display
import ipywidgets as widgets
from threading import Thread
from multiprocessing import Value
import ctypes

# 자동차 초기화
autocar = Pilot.AutoCar()

# 디바이스 설정 (GPU 사용 가능 여부 확인)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 전역 변수: 상태 값 (0: free, 1: blocked)
# 공유 메모리 객체를 사용하여 상태 저장
status_shared = Value(ctypes.c_int, 0)  # 초기값은 free 상태

# 자동차 주행 상태 플래그
driving_flag = Value(ctypes.c_bool, False)  # 초기값은 주행 안함

# 모델 정의
class MyTransferLearningModel(torch.nn.Module):
    def __init__(self, pretrained_model, feature_extractor):
        super().__init__()
        if feature_extractor:
            for param in pretrained_model.parameters():
                param.requires_grad = False
        pretrained_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(pretrained_model.classifier[1].in_features, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(64, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 2)  # 2가지 분류 (blocked, free)
        )
        self.model = pretrained_model
        
    def forward(self, data):
        logits = self.model(data)
        return logits

# 사전 학습된 MobileNetV2 모델 로드
pretrained_model = models.mobilenet_v2(pretrained=True)

# Feature Extractor 여부 설정
feature_extractor = True

# 모델 인스턴스 생성
model = MyTransferLearningModel(pretrained_model, feature_extractor).to(DEVICE)

# 저장된 가중치 파일 경로 (Jetson 환경에 맞게 수정)
MODEL_PATH = "/home/soda/Project/python/notebook/dataset2/MobileNetV2_best_model3.pth"

# 모델 가중치 로드
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()  # 모델을 평가 모드로 설정

# 입력 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNetV2는 224x224 입력을 기대합니다
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
])

# 클래스 레이블 정의
class_labels = ["blocked", "free"]

# 상태 값을 숫자로 변환하는 함수
def get_status_value(state):
    return 1 if state == "blocked" else 0

# 상태 업데이트 함수
def update_status(frame):
    global last_state, current_state, state_start_time, status_shared

    # 모델 입력을 위한 전처리
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # OpenCV 이미지를 PIL 이미지로 변환
    input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)  # 배치 차원 추가 및 GPU로 이동

    # 모델 예측
    with torch.no_grad():  # 그래디언트 계산 비활성화
        outputs = model(input_tensor)
        _, predicted = torch.max(outputs, 1)  # 가장 높은 확률의 클래스 선택

    # 예측 결과 가져오기
    result = class_labels[predicted.item()]

    # 상태 변화 체크
    if result != current_state:
        current_state = result
        state_start_time = time.time()  # 상태 시작 시간 갱신

    # 상태가 유지된 시간 계산
    elapsed_time = time.time() - state_start_time

    # 상태가 0.5초 이상 유지되었을 때만 최종 결과로 간주
    if elapsed_time >= STATE_DURATION_THRESHOLD:
        last_state = current_state

    # 상태 값을 숫자로 변환하고 공유 메모리에 업데이트 (0: free, 1: blocked)
    status_value = get_status_value(last_state)
    with status_shared.get_lock():
        status_shared.value = status_value

    return status_value, last_state

# 카메라 초기화 (224x224 해상도)
cam = Camera(224, 224)

# 상태와 시간 초기화
last_state = "free"  # 이전 상태 초기값 설정
current_state = "free"  # 현재 상태 초기값 설정
state_start_time = time.time()  # 상태 시작 시간
STATE_DURATION_THRESHOLD = 0.5  # 상태가 유지되어야 하는 최소 시간 (초)

# Jupyter Notebook에서 이미지를 표시하기 위한 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

# 상태 출력 텍스트 위젯 생성
status_output = widgets.Text(
    value='Status: 0 (free)',
    description='현재 상태:',
    disabled=True
)
display(status_output)

# 현재 상태값을 가져오는 함수 정의 (다른 셀에서 사용 가능)
def get_current_status():
    with status_shared.get_lock():
        return status_shared.value

# 자동차 상태 제어 함수
def control_car_by_status():
    while True:
        # 0.5초마다 상태 확인
        time.sleep(0.5)
        
        # 상태 확인 (0: free, 1: blocked)
        current_status = get_current_status()
        
        # 주행 중일 때만 상태에 따라 제어
        with driving_flag.get_lock():
            if driving_flag.value:
                if current_status == 1:  # blocked
                    print("장애물 감지! 차량 정지")
                    autocar.steering = 0
                    autocar.stop()
                    # 주행 상태 업데이트
                    driving_flag.value = False

# 카메라 루프 함수
def camera_loop():
    while True:
        # 현재 프레임 읽기
        frame = cam.value
        if frame is None:
            print("프레임을 읽을 수 없습니다.")
            break

        # 상태 업데이트 및 값 반환
        status_value, last_state = update_status(frame)

        # 상태를 화면 상단에 표시
        cv2.putText(frame, f"Status: {last_state} ({status_value})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # 상태 텍스트 위젯 업데이트
        status_output.value = f'Status: {status_value} ({last_state})'

        # Jupyter Notebook에서 프레임 표시
        image_widget.value = cv2.imencode('.jpg', frame)[1].tobytes()
        
        # 화면 갱신을 위한 짧은 대기
        time.sleep(0.03)

# 8자 모양으로 움직이는 함수 정의
def dance_in_8():
    # 주행 상태 설정
    with driving_flag.get_lock():
        driving_flag.value = True
        
    try:
        for _ in range(1):  # 1번 반복 (8자를 그리기 위해)
            # 차량 주행 중 blocked 상태를 감지하면 스레드에서 차량이 정지됨
            
            # 왼쪽으로 회전 (왼쪽 조향 + 전진)
            autocar.steering = -1  # 왼쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(6)          # 6초 동안 유지
            
            # 현재 상태 확인 (차단됐으면 중단)
            if get_current_status() == 1:
                return
            
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(2)          # 2초 동안 유지
            
            # 현재 상태 확인 (차단됐으면 중단)
            if get_current_status() == 1:
                return
            
            # 오른쪽으로 회전 (오른쪽 조향 + 전진)
            autocar.steering = 1   # 오른쪽으로 최대 조향
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(6)          # 6초 동안 유지
            
            # 현재 상태 확인 (차단됐으면 중단)
            if get_current_status() == 1:
                return
            
            autocar.steering = 0  # 직진
            autocar.forward(50)    # 50% 속도로 전진
            time.sleep(1)          # 1초 동안 유지
            
            # 현재 상태 확인 (차단됐으면 중단)
            if get_current_status() == 1:
                return
        
        # 정지
        autocar.steering = 0      # 중앙으로 조향
        autocar.stop()            # 정지
        
    finally:
        # 주행 상태 해제
        with driving_flag.get_lock():
            driving_flag.value = False

# 시작 버튼 위젯 생성
start_button = widgets.Button(
    description="8자 주행 시작",
    button_style="success",
    tooltip="8자 주행을 시작합니다"
)

def on_start_click(b):
    # 별도의 스레드에서 주행 함수 실행
    drive_thread = Thread(target=dance_in_8)
    drive_thread.daemon = True
    drive_thread.start()
    
start_button.on_click(on_start_click)
display(start_button)

print("시스템 초기화 중...")

# 카메라 루프 스레드 시작
camera_thread = Thread(target=camera_loop)
camera_thread.daemon = True
camera_thread.start()

# 자동차 제어 스레드 시작
car_control_thread = Thread(target=control_car_by_status)
car_control_thread.daemon = True
car_control_thread.start()

print("시스템 초기화 완료! '8자 주행 시작' 버튼을 눌러 주행을 시작하세요.")